# Phase 6: LangSmith Observability

This notebook demonstrates:
1. **LangSmith tracing** with custom metadata and tags
2. **Trace inspection** — agent hierarchy, tool calls, latencies
3. **Evaluation dataset** with diverse travel scenarios
4. **Custom evaluators** for domain-specific quality scoring

Prerequisites:
- LangSmith API key in `.env` (`LANGSMITH_API_KEY`)
- `LANGSMITH_TRACING=true` in `.env`
- `LANGSMITH_PROJECT=ai-travel-genie` in `.env`

## 1. Setup

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

# Ensure LangSmith env vars are set
from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "ai-travel-genie")

from langsmith import Client

client = Client()
print(f"LangSmith connected!")
print(f"Project: {os.environ.get('LANGSMITH_PROJECT')}")
print(f"Tracing: {os.environ.get('LANGSMITH_TRACING')}")

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from src.state import TravelPlanState
from src.supervisor import (
    parse_intent, supervisor_router, aggregate_results,
    human_review, after_human_review,
)
from src.agents.destination_agent import destination_research_agent
from src.agents.flight_agent import flight_search_agent
from src.agents.hotel_agent import hotel_search_agent
from src.agents.activity_agent import activity_itinerary_agent
from src.agents.budget_agent import budget_currency_agent

print("All imports successful!")

## 2. Build Graph & Run Traced Trip

We build the same Phase 5 graph. With `LANGSMITH_TRACING=true`, all graph node executions, LLM calls, and tool invocations are automatically traced.

In [ ]:
builder = StateGraph(TravelPlanState)
builder.add_node("intent_parser", parse_intent)
builder.add_node("destination_research", destination_research_agent)
builder.add_node("flight_search", flight_search_agent)
builder.add_node("hotel_search", hotel_search_agent)
builder.add_node("activity_itinerary", activity_itinerary_agent)
builder.add_node("budget_currency", budget_currency_agent)
builder.add_node("aggregator", aggregate_results)
builder.add_node("human_review", human_review)
builder.add_edge(START, "intent_parser")
builder.add_conditional_edges("intent_parser", supervisor_router)
builder.add_conditional_edges("destination_research", supervisor_router)
builder.add_conditional_edges("flight_search", supervisor_router)
builder.add_conditional_edges("hotel_search", supervisor_router)
builder.add_conditional_edges("activity_itinerary", supervisor_router)
builder.add_conditional_edges("budget_currency", supervisor_router)
builder.add_edge("aggregator", "human_review")
builder.add_conditional_edges("human_review", after_human_review)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)
print("Graph compiled with tracing enabled!")

In [ ]:
def safe(s):
    return s.encode('ascii', 'replace').decode('ascii') if isinstance(s, str) else str(s)

# Run a sample trip with tracing
config = {"configurable": {"thread_id": "langsmith-demo-1"}}

print("Running traced trip: Bali Beach Vacation")
print("(All calls will appear in LangSmith dashboard)")
print()

result = graph.invoke(
    {"messages": [HumanMessage(content=(
        "Plan a 5-day beach vacation in Bali for 2 people, "
        "budget $3000, departing from New York. "
        "Dates: 2025-03-15 to 2025-03-20. "
        "Preferences: beach, culture."
    ))]},
    config,
)

print("Graph paused at human_review (traced).")
state = graph.get_state(config)
print(f"Plan status: {state.values.get('plan_status')}")
print(f"All 5 agent outputs populated: {all(state.values.get(f) is not None for f in ['destination_research', 'flight_options', 'hotel_options', 'itinerary', 'budget_summary'])}")

In [ ]:
# Approve the plan to complete the trace
result = graph.invoke(Command(resume="approve"), config)
print(f"Plan status: {graph.get_state(config).values.get('plan_status')}")
print("Trace complete — check LangSmith dashboard!")

## 3. Inspect Traces

Use the LangSmith `Client` to fetch and inspect the trace we just created.

In [ ]:
import time
time.sleep(3)  # brief wait for traces to be indexed

project_name = os.environ.get("LANGSMITH_PROJECT", "ai-travel-genie")

# Fetch recent runs from the project
runs = list(client.list_runs(
    project_name=project_name,
    is_root=True,
    limit=5,
))

print(f"Recent root runs in '{project_name}':")
print()
for run in runs:
    duration = (run.end_time - run.start_time).total_seconds() if run.end_time else 0
    print(f"  Run: {run.name}")
    print(f"    ID: {run.id}")
    print(f"    Status: {run.status}")
    print(f"    Duration: {duration:.1f}s")
    print(f"    Tags: {run.tags}")
    print()

In [ ]:
# Inspect child runs (agent hierarchy) of the most recent root run
if runs:
    root_run = runs[0]
    child_runs = list(client.list_runs(
        project_name=project_name,
        trace_id=root_run.trace_id,
        limit=50,
    ))
    
    print(f"Trace hierarchy for: {root_run.name}")
    print(f"Total spans: {len(child_runs)}")
    print()
    
    # Group by run type
    by_type = {}
    for r in child_runs:
        rt = r.run_type or "unknown"
        by_type.setdefault(rt, []).append(r)
    
    for run_type, type_runs in sorted(by_type.items()):
        print(f"  {run_type} ({len(type_runs)} spans):")
        for r in type_runs[:8]:
            duration = (r.end_time - r.start_time).total_seconds() if r.end_time and r.start_time else 0
            tags = r.tags or []
            print(f"    - {r.name} ({duration:.1f}s) tags={tags}")
        if len(type_runs) > 8:
            print(f"    ... and {len(type_runs) - 8} more")
        print()

## 4. Custom Metadata Demo

Show how our `@traceable` decorators add tags and metadata to each agent run.

In [ ]:
if runs:
    root_run = runs[0]
    
    # Find runs with our custom tags
    agent_runs = list(client.list_runs(
        project_name=project_name,
        trace_id=root_run.trace_id,
        filter='has(tags, "agent")',
        limit=10,
    ))
    
    print("Runs tagged with 'agent':")
    for r in agent_runs:
        duration = (r.end_time - r.start_time).total_seconds() if r.end_time and r.start_time else 0
        print(f"  {r.name}:")
        print(f"    Tags: {r.tags}")
        print(f"    Metadata: {r.extra.get('metadata', {}) if r.extra else {}}")
        print(f"    Duration: {duration:.1f}s")
        print()

    # Find ReAct agent runs
    react_runs = list(client.list_runs(
        project_name=project_name,
        trace_id=root_run.trace_id,
        filter='has(tags, "react")',
        limit=10,
    ))
    
    print("Runs tagged with 'react':")
    for r in react_runs:
        duration = (r.end_time - r.start_time).total_seconds() if r.end_time and r.start_time else 0
        print(f"  {r.name}: {duration:.1f}s, tags={r.tags}")

## 5. Create Evaluation Dataset

Load our 7 evaluation scenarios and push them to LangSmith as a dataset.

In [ ]:
# Load evaluation scenarios
with open(os.path.join("..", "tests", "eval_dataset.json")) as f:
    eval_data = json.load(f)

print(f"Loaded {len(eval_data['scenarios'])} evaluation scenarios:")
for s in eval_data['scenarios']:
    print(f"  - {s['id']}: {s['expected']['destination']} ({s['expected']['days']}d, ${s['expected']['budget_usd']}, {s['expected']['travelers']}pax)")

In [ ]:
# Create or get the LangSmith dataset
dataset_name = eval_data["dataset_name"]

try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"Dataset '{dataset_name}' already exists (id: {dataset.id})")
except Exception:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description=eval_data["description"],
    )
    print(f"Created dataset '{dataset_name}' (id: {dataset.id})")
    
    # Add examples
    for scenario in eval_data["scenarios"]:
        client.create_example(
            inputs={"prompt": scenario["input"]},
            outputs=scenario["expected"],
            dataset_id=dataset.id,
            metadata={"scenario_id": scenario["id"]},
        )
    print(f"Added {len(eval_data['scenarios'])} examples to dataset")

## 6. Define Custom Evaluators

Four domain-specific evaluators that check the quality of the travel plan output.

In [ ]:
def budget_accuracy(run, example) -> dict:
    """Check that budget math is correct: component costs sum to total."""
    try:
        output = run.outputs or {}
        budget = output.get("budget_summary", {})
        if not budget or "total_usd" not in budget:
            return {"key": "budget_accuracy", "score": 0, "comment": "No budget summary"}
        
        components = ["flights", "accommodation", "activities", "food_and_dining", "local_transport", "miscellaneous"]
        comp_sum = sum(budget.get(k, 0) for k in components)
        total = budget.get("total_usd", 0)
        
        accurate = abs(comp_sum - total) < 1
        return {
            "key": "budget_accuracy",
            "score": 1.0 if accurate else 0.0,
            "comment": f"Sum={comp_sum}, Total={total}, diff={abs(comp_sum - total):.2f}"
        }
    except Exception as e:
        return {"key": "budget_accuracy", "score": 0, "comment": str(e)}


def destination_correctness(run, example) -> dict:
    """Check that the parsed destination matches expected."""
    try:
        output = run.outputs or {}
        expected = example.outputs or {}
        
        parsed_dest = output.get("travel_request", {}).get("destination", "").lower()
        expected_dest = expected.get("destination", "").lower()
        
        correct = expected_dest in parsed_dest or parsed_dest in expected_dest
        return {
            "key": "destination_correctness",
            "score": 1.0 if correct else 0.0,
            "comment": f"Parsed: {parsed_dest}, Expected: {expected_dest}"
        }
    except Exception as e:
        return {"key": "destination_correctness", "score": 0, "comment": str(e)}


def itinerary_completeness(run, example) -> dict:
    """Check day count, AM/PM/EVE slots, and weather in itinerary."""
    try:
        output = run.outputs or {}
        expected = example.outputs or {}
        
        itinerary = output.get("itinerary", {})
        days = itinerary.get("days", [])
        expected_days = expected.get("days", 0)
        
        day_count_ok = len(days) == expected_days
        slots_ok = all(
            "morning" in d and "afternoon" in d and "evening" in d and "weather" in d
            for d in days
        ) if days else False
        
        score = (0.5 if day_count_ok else 0) + (0.5 if slots_ok else 0)
        return {
            "key": "itinerary_completeness",
            "score": score,
            "comment": f"Days: {len(days)}/{expected_days}, Slots: {'OK' if slots_ok else 'MISSING'}"
        }
    except Exception as e:
        return {"key": "itinerary_completeness", "score": 0, "comment": str(e)}


def agent_output_completeness(run, example) -> dict:
    """Check that all required agent outputs are populated."""
    try:
        output = run.outputs or {}
        expected = example.outputs or {}
        
        required = expected.get("required_agent_outputs", [])
        populated = [f for f in required if output.get(f) is not None]
        
        score = len(populated) / len(required) if required else 0
        missing = [f for f in required if f not in populated]
        return {
            "key": "agent_output_completeness",
            "score": score,
            "comment": f"{len(populated)}/{len(required)} outputs populated" + (f", missing: {missing}" if missing else "")
        }
    except Exception as e:
        return {"key": "agent_output_completeness", "score": 0, "comment": str(e)}


print("4 custom evaluators defined:")
print("  - budget_accuracy: budget math check")
print("  - destination_correctness: intent parsing check")
print("  - itinerary_completeness: day count + slot check")
print("  - agent_output_completeness: all agents produced output")

## 7. Run Evaluation

Run the graph against each evaluation scenario and score the results.

**Note**: This runs each scenario through the full 5-agent pipeline (with LLM calls). Each scenario takes 2-4 minutes.

In [ ]:
# Build a non-HIL graph for evaluation (no human_review interrupt)
eval_builder = StateGraph(TravelPlanState)
eval_builder.add_node("intent_parser", parse_intent)
eval_builder.add_node("destination_research", destination_research_agent)
eval_builder.add_node("flight_search", flight_search_agent)
eval_builder.add_node("hotel_search", hotel_search_agent)
eval_builder.add_node("activity_itinerary", activity_itinerary_agent)
eval_builder.add_node("budget_currency", budget_currency_agent)
eval_builder.add_node("aggregator", aggregate_results)
eval_builder.add_edge(START, "intent_parser")
eval_builder.add_conditional_edges("intent_parser", supervisor_router)
eval_builder.add_conditional_edges("destination_research", supervisor_router)
eval_builder.add_conditional_edges("flight_search", supervisor_router)
eval_builder.add_conditional_edges("hotel_search", supervisor_router)
eval_builder.add_conditional_edges("activity_itinerary", supervisor_router)
eval_builder.add_conditional_edges("budget_currency", supervisor_router)
eval_builder.add_edge("aggregator", END)
eval_graph = eval_builder.compile()

print("Evaluation graph compiled (no human_review interrupt)")

In [ ]:
# Run evaluation on 2 scenarios (to keep time reasonable)
eval_scenarios = eval_data["scenarios"][:2]  # Bali + Maldives

results = []
for scenario in eval_scenarios:
    print(f"\nEvaluating: {scenario['id']}...")
    
    output = eval_graph.invoke(
        {"messages": [HumanMessage(content=scenario["input"])]}
    )
    
    # Create a mock run/example for our evaluators
    class MockObj:
        def __init__(self, d):
            self.outputs = d
    
    mock_run = MockObj(output)
    mock_example = MockObj(scenario["expected"])
    
    scores = {
        "scenario": scenario["id"],
        "budget_accuracy": budget_accuracy(mock_run, mock_example),
        "destination_correctness": destination_correctness(mock_run, mock_example),
        "itinerary_completeness": itinerary_completeness(mock_run, mock_example),
        "agent_output_completeness": agent_output_completeness(mock_run, mock_example),
    }
    results.append(scores)
    
    print(f"  Budget accuracy:    {scores['budget_accuracy']['score']:.1f} - {scores['budget_accuracy']['comment']}")
    print(f"  Destination:        {scores['destination_correctness']['score']:.1f} - {scores['destination_correctness']['comment']}")
    print(f"  Itinerary:          {scores['itinerary_completeness']['score']:.1f} - {scores['itinerary_completeness']['comment']}")
    print(f"  Agent completeness: {scores['agent_output_completeness']['score']:.1f} - {scores['agent_output_completeness']['comment']}")

In [ ]:
# Summary
print("=" * 70)
print("  EVALUATION SUMMARY")
print("=" * 70)
print()

evaluators = ["budget_accuracy", "destination_correctness", "itinerary_completeness", "agent_output_completeness"]

for ev in evaluators:
    scores = [r[ev]["score"] for r in results]
    avg = sum(scores) / len(scores)
    print(f"  {ev:30s}  avg={avg:.2f}  scores={scores}")

overall = sum(
    r[ev]["score"] for r in results for ev in evaluators
) / (len(results) * len(evaluators))
print(f"\n  Overall average: {overall:.2f}")

## 8. Summary

### What we demonstrated:
1. **Tracing**: All graph nodes, LLM calls, and tool invocations are traced to LangSmith
2. **Custom metadata**: `@traceable` decorators add agent-specific tags (`agent`, `react`, `supervisor`) and metadata (`phase`)
3. **Trace inspection**: Used `Client.list_runs()` to fetch and analyze trace hierarchy
4. **Evaluation dataset**: 7 diverse scenarios pushed to LangSmith
5. **Custom evaluators**: 4 domain-specific quality checks (budget math, destination parsing, itinerary slots, agent completeness)

### What to look for in LangSmith:
- **Project**: `ai-travel-genie`
- **Trace view**: Click any run to see the full agent hierarchy — each node (intent_parser, destination_research, etc.) appears as a child span
- **Tag filtering**: Filter by `agent`, `react`, `supervisor` tags
- **Latency**: Compare agent execution times to find bottlenecks
- **Tool calls**: Each `@tool` invocation (search_flights, search_hotels, etc.) appears as a child of its agent's ReAct loop

### Next: Phase 7 (Polish & Presentation)